In [2]:
import duckdb

In [23]:
%load_ext sql
%sql duckdb://
%sql INSTALL excel; LOAD excel;

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Running query in 'duckdb://'

Success


In [40]:
%sql ROLLBACK;

Running query in 'duckdb://'

Success


In [ ]:
%%sql
-- Report 1: Open Requests Log
WITH customer_care AS (
    SELECT
        *,
        CAST("Date of Submission" AS DOUBLE) AS submission_serial
    FROM read_xlsx(
        'Customer Care.xlsx',
        header = true,
        all_varchar = true
    )
),
cleaned AS (
    SELECT
        *,
        CAST(
            DATE '1899-12-30'
            + CAST(FLOOR(submission_serial) AS INTEGER)
            AS TIMESTAMP
        )
        + (
            submission_serial - FLOOR(submission_serial)
        ) * INTERVAL '1 day' AS submission_datetime
    FROM customer_care
)
SELECT
    "Response Id" AS response_id,
    submission_datetime AS date_of_submission,
    "Region" AS region,
    "Zone" AS zone,
    "Section" AS section,
    "Famer Name." AS farmer_name,
    "Farmer Phone No." AS farmer_phone,
    "Requests" AS request_type,
    COALESCE(
        NULLIF(TRIM("Request Details"), ''),
        'Not Started'
    ) AS request_details,
    COALESCE(
        NULLIF(TRIM("RHOA STATUS"), ''),
        'Not Started'
    ) AS rhoa_status,
    COALESCE(
        NULLIF(TRIM("REMARKS AT CUSTOMER CARE"), ''),
        'Not Started'
    ) AS remarks_customer_care,
    DATE_DIFF(
        'day',
        CAST(submission_datetime AS DATE),
        CURRENT_DATE
    ) AS days_open
FROM cleaned
WHERE "RHOA STATUS" IS NULL
   OR TRIM("RHOA STATUS") = ''
   OR "RHOA STATUS" IN ('Not Started', 'In Progress')
ORDER BY days_open DESC;

Running query in 'duckdb://'

response_id,date_of_submission,region,zone,section,farmer_name,farmer_phone,request_type,request_details,rhoa_status,remarks_customer_care,days_open
bb0d29f4,2026-09-03 12:19:16,KABRAS B,SOUTHERN ZONE-A,SOUTH-A 4,Paul Amanza kikin,7.1657758E8,Permit Mill Cane,Not Started,Not Started,Not Started,14
163e41ad,2026-09-03 15:59:49,KABRAS B,KAIMOSI,KAIMOSI 3,Hellen kitur,7.28148585E8,Permit Mill Cane,Not Started,Not Started,Not Started,14
fbc3bbc4,2026-09-03 12:23:32,BUTERE,SHIANDA ZONE,SHIANDA-3,Josephat Opworah,7.25703894E8,Permit Mill Cane,Not Started,Not Started,Not Started,14
1a873300,2026-09-03 12:25:22,BUTERE,SHIANDA ZONE,SHIANDA-2,Faith Mwaro,7.26795699E8,Permit Mill Cane,Not Started,Not Started,Not Started,14
7c6b73a0,2026-09-03 12:28:16,KABRAS B,EASTERN ZONE,EAST 5,Symon khayega,7.11325354E8,Fertilizer,Not Started,Not Started,Not Started,14
43feab17,2026-09-03 12:29:57,KHALABA,NAMULUNGU,NAMULUNGU 1,GEORGE OMULAMA NETIA,7.17607895E8,Permit Seed Cane,Not Started,Not Started,Not Started,14
f98a3028,2026-09-03 12:30:38,KABRAS B,SHINYALU,SHINYALU-1,Aggrey Shujaa lilumbi,7.25825433E8,Seed Cane,Not Started,Not Started,Not Started,14
c375ddb1,2026-09-03 12:35:24,KABRAS A,WESTERN ZONE-D,WEST-D 1,Jotham kundu Matala,7.15753083E8,Fertilizer,Not Started,Not Started,Not Started,14
29936c2b,2026-09-03 12:39:02,KABRAS A,WESTERN ZONE-A,WEST-A 5,Milcah Osimbo Liwa,7.1072972E8,Filtermud (Only Appoved Areas),Not Started,Not Started,Not Started,14
5.4191728E7,2026-09-03 12:40:06,KHALABA,BULONDO,BULONDO-1,Boaz Juma Mayende,7.9916514E8,Permit Mill Cane,Not Started,Not Started,Not Started,14


In [52]:
%%sql
-- Report 2: Resolved Requests Log
WITH customer_care AS (
    SELECT
        *,
        CAST("Date of Submission" AS DOUBLE) AS submission_serial
    FROM read_xlsx(
        'Customer Care.xlsx',
        header = true,
        all_varchar = true
    )
),
cleaned AS (
    SELECT
        *,
        CAST(
            DATE '1899-12-30'
            + CAST(FLOOR(submission_serial) AS INTEGER)
            AS TIMESTAMP
        )
        + (
            submission_serial - FLOOR(submission_serial)
        ) * INTERVAL '1 day' AS submission_datetime
    FROM customer_care
)
SELECT
    "Response Id" AS response_id,
    submission_datetime AS date_of_submission,
    "Region" AS region,
    "Zone" AS zone,
    "Famer Name." AS farmer_name,
    "Requests" AS request_type,
    COALESCE(
        NULLIF(TRIM("CLOSED THROUGH FIELD NUMBER"), ''),
        'Not Started'
    ) AS closed_through_field_number,
    COALESCE(
        NULLIF(TRIM("RHOA REMARKS"), ''),
        'Not Started'
    ) AS rhoa_remarks
FROM cleaned
WHERE TRIM("RHOA STATUS") = 'Converted'
ORDER BY date_of_submission DESC;

Running query in 'duckdb://'

response_id,date_of_submission,region,zone,farmer_name,request_type,closed_through_field_number,rhoa_remarks


In [ ]:
%%sql
-- Report 3: Regional Summary
WITH customer_care AS (
    SELECT
        "Region" AS region,
        "RHOA STATUS" AS rhoa_status
    FROM read_xlsx(
        'Customer Care.xlsx',
        header = true,
        all_varchar = true
    )
)
SELECT
    TRIM(region) AS region,
    COUNT(*) AS total_requests,
    SUM(CASE WHEN TRIM(rhoa_status) = 'Converted' THEN 1 ELSE 0 END) AS resolved,
    SUM(
        CASE
            WHEN rhoa_status IS NULL
              OR TRIM(rhoa_status) = ''
              OR TRIM(rhoa_status) <> 'Converted'
            THEN 1
            ELSE 0
        END
    ) AS pending,
    ROUND(
        100.0 * SUM(
            CASE WHEN TRIM(rhoa_status) = 'Converted' THEN 1 ELSE 0 END
        ) / NULLIF(COUNT(*), 0),
        1
    ) AS resolved_rate_pct
FROM customer_care
GROUP BY TRIM(region)
ORDER BY total_requests DESC;

Running query in 'duckdb://'

region,total_requests,resolved,pending,resolved_rate_pct
KABRAS A,262,0,262,0.0
KABRAS B,228,0,228,0.0
MAGUT,78,0,78,0.0
BUTERE,75,0,75,0.0
SELIA,71,0,71,0.0
KHALABA,50,0,50,0.0
BUSIA,48,0,48,0.0
MISIKHU,37,0,37,0.0
NAITIRI,20,0,20,0.0
None,2,0,2,0.0


In [50]:
%%sql
-- Report 4: Farmer Request History

WITH customer_care AS (
    SELECT
        "Response Id" AS farmer_id,
        REGEXP_REPLACE(TRIM("Famer Name."), '[\r\n]+', ' ', 'g') AS farmer_name,
        TRIM("Farmer Phone No.") AS farmer_phone_no,
        REGEXP_REPLACE(TRIM("Region"), '[\r\n]+', ' ', 'g') AS region,
        TRIM("Requests") AS requests,
        CAST("Date of Submission" AS DOUBLE) AS submission_serial
    FROM read_xlsx(
        'Customer Care.xlsx',
        header = true,
        all_varchar = true
    )
),

cleaned AS (
    SELECT
        *,
        CAST(
            DATE '1899-12-30'
            + CAST(FLOOR(submission_serial) AS INTEGER)
            AS TIMESTAMP
        )
        + (
            submission_serial - FLOOR(submission_serial)
        ) * INTERVAL '1 day' AS submission_datetime
    FROM customer_care
)

SELECT
    farmer_id,
    farmer_name,
    farmer_phone_no,
    region,
    COUNT(*) AS total_requests,
    STRING_AGG(requests, ', ') AS request_types,
    MAX(submission_datetime) AS most_recent_request
FROM cleaned
GROUP BY
    farmer_id,
    farmer_name,
    farmer_phone_no,
    region
ORDER BY total_requests DESC;

Running query in 'duckdb://'

farmer_id,farmer_name,farmer_phone_no,region,total_requests,request_types,most_recent_request
fbc3bbc4,Josephat Opworah,7.25703894E8,BUTERE,1,Permit Mill Cane,2026-09-03 12:23:32
13cb0903,Stanely Kering,7.40965549E8,SELIA,1,Fertilizer,2026-09-03 13:42:39
0730d98f,Benerd Odhiamba,7.1586035E8,KABRAS B,1,Permit Mill Cane,2026-09-03 14:09:23
7f15ba8a,Charles Simiyu,7.95193702E8,KABRAS A,1,Filtermud (Only Appoved Areas),2026-09-03 14:15:52
16a73c33,Josephat Mulefu Muzee,7.94432384E8,KABRAS A,1,Permit Mill Cane,2026-09-03 14:18:43
8144c083,Daniel Muyaka Welakoko,7.68681369E8,KABRAS A,1,Contracting (Registration),2026-09-03 14:30:25
53bd3a08,Jotham matala,7.15753083E8,KABRAS A,1,Fertilizer,2026-09-03 14:52:19
b1b87451,Rose Luhazo,7.17686015E8,KABRAS B,1,Seed Cane,2026-09-03 15:13:31
ad6ce238,Beatrice Lusiche,7.95564702E8,KABRAS A,1,Permit Mill Cane,2026-09-03 15:35:49
d7953a96,Maureen Mweyi,7.20175299E8,MAGUT,1,Seed Cane,2026-09-03 15:38:21
